
# Level 2 L1000 (raw landmark) → DEG-ready pseudobulk

This notebook mirrors `notebooks/l1000_deg_prep.ipynb` but uses the raw **Level 2** epsilon release (`GSE92742`). It documents every download, enriches metadata (cell, donor, perturbagen), and emits an `AnnData` that matches the 24-column schema described in `op3_v2/docs/Format_Pseudobulk.ipynb`.

Key differences vs. the Level 3 workflow:
- A single GCTX (`GSE92742_Broad_LINCS_Level2_GEX_epsilon_n1269922x978.gctx.gz`) holds all perturbation types; filtering happens here instead of reading multiple shards.
- Expression values are raw landmark intensities (no inference, no normalization). Downstream DEG scripts can decide how/if to normalize.
- The notebook auto-builds `compoundinfo_beta_with_MW.tsv` using RDKit when missing, so you always have molecular weights for unit conversions.



## Required `.obs` schema

Same 24-column structure as the pseudobulk validator. Anything extra should be dropped before writing to disk.


### <span style="color:blue;">*Comment 1* </span>
<span style="color:blue;"> *Added path to the extra cell line info file `cellinfo_beta.txt`.*</span>

In [1]:

from __future__ import annotations

import json
import subprocess
from collections import OrderedDict
from pathlib import Path
from typing import Iterable, Optional


import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)

PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = (PROJECT_ROOT / "lincs_data").resolve()
DATA_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = DATA_ROOT / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)


PATHS = OrderedDict({
    "level2_gctx": DATA_ROOT / "GSE92742_Broad_LINCS_Level2_GEX_epsilon_n1269922x978.gctx",
    "level2_gctx_gz": DATA_ROOT / "GSE92742_Broad_LINCS_Level2_GEX_epsilon_n1269922x978.gctx.gz",
    "instinfo": DATA_ROOT / "GSE92742_Broad_LINCS_inst_info.txt",
    "cellinfo": DATA_ROOT / "GSE92742_Broad_LINCS_cell_info.txt",
    "pert_info": DATA_ROOT / "GSE92742_Broad_LINCS_pert_info.txt",
    "geneinfo_level2": DATA_ROOT / "GSE92742_Broad_LINCS_gene_info.txt",
    "geneinfo_beta": DATA_ROOT / "geneinfo_beta.txt",
    "cellinfo_beta": DATA_ROOT / "cellinfo_beta.txt",
    "compoundinfo_mw": DATA_ROOT / "compoundinfo_beta_with_MW.tsv",
})

AUTO_GENERATED = {"compoundinfo_mw"}

OBS_SCHEMA = [
    ("sample_id", "category", "ID of the observation: plate + well + cell_type + perturbagen"),
    ("plate", "category", "Assay plate identifier (det_plate)"),
    ("well", "category", "Well ID on the RNA plate (rna_well)"),
    ("cell_type", "category", "Cell line / cell_id"),
    ("perturbagen", "category", "Human-readable perturbagen label"),
    ("pert_type", "category", "Perturbation class"),
    ("is_control", "category", "True/False for controls"),
    ("pert_dose_uM", "float64", "Dose in micromolar"),
    ("pert_time_h", "float64", "Exposure time in hours"),
    ("suspension_type", "category", "Growth pattern"),
    ("tissue", "category", "Primary tissue/site"),
    ("tissue_type", "category", "Sample type"),
    ("disease", "category", "Disease/subtype"),
    ("library", "category", "Library/release"),
    ("stimulation", "category", "High-level stimulus"),
    ("guide", "category", "A guide RNA directs the CRISPR system"),
    ("dataset", "category", "Dataset label"),
    ("assay", "category", "Assay label"),
    ("development_stage", "category", "Derived from donor age"),
    ("organism", "category", "Organism"),
    ("sex", "category", "Donor sex"),
    ("self_reported_ethnicity", "category", "Donor ethnicity"),
    ("pubchem_cid", "category", "PubChem CID"),
    ("psbulk_cells", "int64", "Total #cells contributing (if no info - then -666)"),
    ("psbulk_counts", "int64", "Total #counts contributing (if no info - then -666)"),
]


In [2]:
obs_schema_df = pd.DataFrame(OBS_SCHEMA, columns=["column", "dtype", "description"]).set_index("column")
obs_schema_df

,dtype,description
column,,
sample_id,category,ID of the observation: plate + well + cell_typ...
plate,category,Assay plate identifier (det_plate)
well,category,Well ID on the RNA plate (rna_well)
cell_type,category,Cell line / cell_id
perturbagen,category,Human-readable perturbagen label
pert_type,category,Perturbation class
is_control,category,True/False for controls
pert_dose_uM,float64,Dose in micromolar
pert_time_h,float64,Exposure time in hours



## Downloads & checksums

Level-2 files live on GEO’s FTP server. Metadata companions (inst, cell, pert, gene) come from the same drop, while `geneinfo_beta.txt` is from the 2020 CLUE beta release and is used to map Ensembl IDs.

`compoundinfo_beta_with_MW.tsv` is generated automatically later, so no download is required for it.


### <span style="color:blue;">*Comment 2* </span>
<span style="color:blue;"> *I just corrected downloading of the sources here and unpacking of archived files. Also added downloading extra cell line info,  `cellinfo_beta.txt`, for retrieving Cellosaurus IDs.*</span>

In [3]:

download_manifest = [
    {"file": "GSE92742_Broad_LINCS_Level2_GEX_epsilon_n1269922x978.gctx.gz", "kind": "expression", "size": "2.3 GB",
     "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_Level2_GEX_epsilon_n1269922x978.gctx.gz",
     "path": PATHS["level2_gctx_gz"], "notes": "Raw epsilon (landmark genes)"},
    {"file": "GSE92742_Broad_LINCS_inst_info.txt.gz", "kind": "metadata", "size": "~150 MB",
     "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_inst_info.txt.gz",
     "path": PATHS["instinfo"], "notes": "Instance-level annotations"},
    {"file": "GSE92742_Broad_LINCS_cell_info.txt.gz", "kind": "metadata", "size": "<10 KB",
     "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_cell_info.txt.gz",
     "path": PATHS["cellinfo"], "notes": "Cell line annotations"},
    {"file": "GSE92742_Broad_LINCS_pert_info.txt.gz", "kind": "metadata", "size": "~5 MB",
     "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_pert_info.txt.gz",
     "path": PATHS["pert_info"], "notes": "Perturbagen metadata"},
    {"file": "GSE92742_Broad_LINCS_gene_info.txt.gz", "kind": "metadata", "size": "~210 KB",
     "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_gene_info.txt.gz",
     "path": PATHS["geneinfo_level2"], "notes": "Landmark gene annotations"},
    {"file": "geneinfo_beta.txt", "kind": "metadata", "size": "1.09 MB",
     "url": "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/geneinfo_beta.txt",
     "path": PATHS["geneinfo_beta"], "notes": "2020 CLUE gene dictionary (for Ensembl IDs)"},
    {"file": "cellinfo_beta.txt", "kind": "metadata", "size": "0.04 MB",
     "url": "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/cellinfo_beta.txt",
     "path": PATHS["cellinfo_beta"], "notes": "2020 CLUE Cell line annotations (for Cellosaurus IDs)"},
]

manifest_df = pd.DataFrame(download_manifest)
manifest_df["curl_example"] = manifest_df.apply(
    lambda row: (
        f"curl -L '{row['url']}' -o {DATA_ROOT / row['file']}"
        if row["url"].startswith("http")
        else (
            f"curl -p --insecure '{row['url']}' -o {DATA_ROOT / row['file']}\n"
            f"# or wget '{row['url']}'"
        )
    ),
    axis=1,
)
manifest_df[["file", "kind", "size", "notes", "curl_example"]]


,file,kind,size,notes,curl_example
0,GSE92742_Broad_LINCS_Level2_GEX_epsilon_n12699...,expression,2.3 GB,Raw epsilon (landmark genes),curl -L 'https://ftp.ncbi.nlm.nih.gov/geo/seri...
1,GSE92742_Broad_LINCS_inst_info.txt.gz,metadata,~150 MB,Instance-level annotations,curl -L 'https://ftp.ncbi.nlm.nih.gov/geo/seri...
2,GSE92742_Broad_LINCS_cell_info.txt.gz,metadata,<10 KB,Cell line annotations,curl -L 'https://ftp.ncbi.nlm.nih.gov/geo/seri...
3,GSE92742_Broad_LINCS_pert_info.txt.gz,metadata,~5 MB,Perturbagen metadata,curl -L 'https://ftp.ncbi.nlm.nih.gov/geo/seri...
4,GSE92742_Broad_LINCS_gene_info.txt.gz,metadata,~210 KB,Landmark gene annotations,curl -L 'https://ftp.ncbi.nlm.nih.gov/geo/seri...
5,geneinfo_beta.txt,metadata,1.09 MB,2020 CLUE gene dictionary (for Ensembl IDs),curl -L 'https://s3.amazonaws.com/macchiato.cl...
6,cellinfo_beta.txt,metadata,0.04 MB,2020 CLUE Cell line annotations (for Cellosaur...,curl -L 'https://s3.amazonaws.com/macchiato.cl...


In [4]:
for row in manifest_df.values:
    if not (DATA_ROOT / row[0]).exists():
        cmd = row[-1].split('or')[0].strip(' #\n')
        subprocess.call(cmd, shell=True)

In [5]:
missing = []
compressed = []
ready = []

for key, path in PATHS.items():
    if key in AUTO_GENERATED:
        continue
    # Skip the compressed file keys (they are the source files)
    if key.endswith("_gz"):
        continue
    
    # Check if file exists (uncompressed or compressed)
    if path.suffix == ".gz":
        if not path.exists():
            missing.append((key, path))
    else:
        gz_candidate = path.with_suffix(path.suffix + ".gz")
        if path.exists():
            ready.append((key, path))
        elif gz_candidate.exists():
            compressed.append((key, gz_candidate))
        else:
            missing.append((key, path))

if missing:
    print("⚠️ Missing files (download as per manifest):")
    for key, path in missing:
        print(f"  - {key}: {path}")
else:
    print("✅ All required downloads are present.")

if compressed:
    print(f"\n⚠️ {len(compressed)} file(s) still compressed (needs extraction):")
    for key, path in compressed:
        print(f"  - {key}: {path.name}")
    if any(key == "level2_gctx" for key, _ in compressed):
        print("  → Run the next cell to extract GCTX (needs ~5 GB free).")
else:
    print("✅ All files are uncompressed and ready to use.")


✅ All required downloads are present.
✅ All files are uncompressed and ready to use.



### Optional: decompress the `.gctx.gz`

`cmapPy` cannot read gzipped GCTX files directly. Use this helper (or `gunzip` on the command line) to extract the matrix.


In [6]:

import gzip
import shutil

to_decompress = []
already_done = []

for key, path in PATHS.items():
    if key in AUTO_GENERATED:
        continue
    # Skip the compressed file keys (they are the source files)
    if key.endswith("_gz"):
        continue
    
    # Check if file needs decompression
    if not path.suffix == ".gz":
        gz_candidate = path.with_suffix(path.suffix + ".gz")
        if gz_candidate.exists() and not path.exists():
            to_decompress.append((key, gz_candidate, path))
        elif path.exists():
            already_done.append((key, path))

if already_done:
    print(f"✅ {len(already_done)} file(s) already decompressed:")
    for key, path in already_done:
        print(f"  - {key}: {path.name}")

if to_decompress:
    print(f"\n🔄 Decompressing {len(to_decompress)} file(s)...")
    for key, gz_path, target_path in to_decompress:
        print(f"  - {key}: {gz_path.name} → {target_path.name}")
        if "gctx" in key.lower():
            print("    (GCTX is large, this may take a few minutes...)")
        
        with gzip.open(gz_path, "rb") as src, open(target_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
        print(f"    ✅ Done")
    
    print(f"\n✅ All {len(to_decompress)} file(s) successfully decompressed!")
else:
    if not already_done:
        print("⚠️ No compressed files found. Download them via the manifest first.")
    else:
        print("\n✅ All files are already decompressed, nothing to do.")


✅ 7 file(s) already decompressed:
  - level2_gctx: GSE92742_Broad_LINCS_Level2_GEX_epsilon_n1269922x978.gctx
  - instinfo: GSE92742_Broad_LINCS_inst_info.txt
  - cellinfo: GSE92742_Broad_LINCS_cell_info.txt
  - pert_info: GSE92742_Broad_LINCS_pert_info.txt
  - geneinfo_level2: GSE92742_Broad_LINCS_gene_info.txt
  - geneinfo_beta: geneinfo_beta.txt
  - cellinfo_beta: cellinfo_beta.txt

✅ All files are already decompressed, nothing to do.



## Configure subsets (keep iteration fast)

Prototype with aggressive filters (few cell lines / compounds / samples) so the memmap step completes quickly. Once the flow looks good, remove the caps to rebuild the full dataset (~1.27 M profiles).


### <span style="color:blue;">*Comment 3* </span>
<span style="color:blue;"> *I might be wrong, but as I see, it might be difficult to run the full dataset in the one run as it contains the mixture of different perturbation types. Maybe it would be a bit easier to take just compound perturbations and run deg pipeline on them as the prototype, as currently we have already tested our pipeline on the perturbations with compounds. Once we complete it, then we can switch to other types of perturbations and expand our pipeline.*</span>

In [7]:
'''
CONFIG = {
    "pert_types_to_keep": {"trt_cp", "ctl_vehicle", "ctl_untrt", "ctl_vector"},
    "cell_types_to_keep": None,
    "perturbagens_to_keep": None,
    "only_touchstone": False,
    "full_gene_matrix": False,
    # "max_samples": None,
    "max_samples": 5000,
    "min_dose_uM": None,
    "max_dose_uM": None,
    "time_h_min": None,
    "time_h_max": None,
}

ASSAY_LABEL = "L1000"
DATASET_LABEL = "LINCS2017_level2"
LIBRARY_LABEL = "LINCS_L1000_Level2_epsilon"
'''

'\nCONFIG = {\n    "pert_types_to_keep": {"trt_cp", "ctl_vehicle", "ctl_untrt", "ctl_vector"},\n    "cell_types_to_keep": None,\n    "perturbagens_to_keep": None,\n    "only_touchstone": False,\n    "full_gene_matrix": False,\n    # "max_samples": None,\n    "max_samples": 5000,\n    "min_dose_uM": None,\n    "max_dose_uM": None,\n    "time_h_min": None,\n    "time_h_max": None,\n}\n\nASSAY_LABEL = "L1000"\nDATASET_LABEL = "LINCS2017_level2"\nLIBRARY_LABEL = "LINCS_L1000_Level2_epsilon"\n'

<span style="color:blue;"> *Therefore I would change a config file a bit: to include just `DMSO` as `ctl_vehicle` and `trt_cp`:*</span>

In [8]:
CONFIG = {
    "pert_types_to_keep": {"trt_cp", "ctl_vehicle"},
    "ctl_perturbagens": {"DMSO"},
    "cell_types_to_keep": None,
    "perturbagens_to_keep": None,
    "only_touchstone": False,
    "full_gene_matrix": False,
    "max_samples": None,
    #"max_samples": 5000,
    "min_dose_uM": None,
    "max_dose_uM": None,
    "time_h_min": None,
    "time_h_max": None,
}

ASSAY_LABEL = "L1000"
DATASET_LABEL = "LINCS2017_level2"
LIBRARY_LABEL = "LINCS_L1000_Level2_epsilon"

In [9]:
FULL_GENE_MATRIX = CONFIG.get("full_gene_matrix", True)
GENE_SCOPE_TAG = "full" if FULL_GENE_MATRIX else "landmark"

OUT_H5AD = PROCESSED_DIR / ("l1000_level2_deg_ready.h5ad" if FULL_GENE_MATRIX else f"l1000_level2_deg_ready_{GENE_SCOPE_TAG}.h5ad")
EXPR_MEMMAP = PROCESSED_DIR / ("l1000_level2_expr.npy" if FULL_GENE_MATRIX else f"l1000_level2_expr_{GENE_SCOPE_TAG}.npy")
EXPR_SAMPLE_IDS = PROCESSED_DIR / "l1000_level2_sample_ids.npy"
EXPR_GENE_IDS = PROCESSED_DIR / ("l1000_level2_gene_ids.npy" if FULL_GENE_MATRIX else f"l1000_level2_gene_ids_{GENE_SCOPE_TAG}.npy")


## Load metadata tables

In [10]:

def _read_table(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        gz = path.with_suffix(path.suffix + ".gz")
        if gz.exists():
            import gzip
            with gzip.open(gz, "rt") as fh:
                df = pd.read_csv(fh, **kwargs)
        else:
            raise FileNotFoundError(path)
    else:
        df = pd.read_csv(path, **kwargs)
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.lower()
    )
    return df

inst_raw = _read_table(PATHS["instinfo"], sep="	", low_memory=False)
cellinfo_raw = _read_table(PATHS["cellinfo"], sep="	")
pert_raw = _read_table(PATHS["pert_info"], sep="	")
geneinfo_level2 = _read_table(PATHS["geneinfo_level2"], sep="	")
geneinfo_beta = _read_table(PATHS["geneinfo_beta"], sep="	") if PATHS["geneinfo_beta"].exists() else None
cellinfo_beta = _read_table(PATHS["cellinfo_beta"], sep="	") if PATHS["cellinfo_beta"].exists() else None

print(f"inst rows: {len(inst_raw):,}")
print(f"cell rows: {len(cellinfo_raw):,}")
print(f"pert rows: {len(pert_raw):,}")
print(f"gene rows (landmark): {len(geneinfo_level2):,}")


inst rows: 1,319,138
cell rows: 98
pert rows: 51,383
gene rows (landmark): 12,328



### Auto-build `compoundinfo_beta_with_MW.tsv`

We derive molecular weights directly from `pert_info` using RDKit (needs `rdkit-pypi` or conda `rdkit`). The resulting file is saved to `PATHS["compoundinfo_mw"]` for reuse.


In [11]:
def ensure_compoundinfo_with_mw(pert_path: Path, output_path: Path) -> Path:
    if output_path.exists():
        print("✔ compoundinfo with MW already present →", output_path)
        return output_path
    try:
        from rdkit import Chem
        from rdkit.Chem import Descriptors
    except ImportError as exc:
        raise ImportError(
            "rdkit is required to build compoundinfo_beta_with_MW.tsv. Install via `pip install rdkit-pypi` or conda rdkit."
        ) from exc

    df = _read_table(pert_path, sep="	")
    df = df.rename(columns={"pert_iname": "compound_name"})

    def smiles_to_mw(smiles: str) -> float:
        smiles = str(smiles)
        if not smiles or smiles in {"-666", "nan", "None"}:
            return np.nan
        mol = Chem.MolFromSmiles(smiles)
        return Descriptors.ExactMolWt(mol) if mol is not None else np.nan

    df["mw_g_per_mol"] = df["canonical_smiles"].map(smiles_to_mw)
    cols = ["pert_id", "compound_name", "canonical_smiles", "inchi_key", "compound_aliases", "mw_g_per_mol"]
    available_cols = [c for c in cols if c in df.columns]
    out_df = df[available_cols].drop_duplicates(subset=["pert_id"]) if "pert_id" in df.columns else df[available_cols]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(output_path, sep="	", index=False)
    print("✅ Built", output_path)
    return output_path

compoundinfo_path = ensure_compoundinfo_with_mw(PATHS["pert_info"], PATHS["compoundinfo_mw"])
compound_mw = _read_table(compoundinfo_path, sep="	")
print(f"compound rows with MW: {compound_mw['mw_g_per_mol'].notna().sum():,}")


✔ compoundinfo with MW already present → /ictstr01/home/icb/olga.novitskaia/lincs_data/compoundinfo_beta_with_MW.tsv
compound rows with MW: 20,337


## Helper functions

### <span style="color:blue;">*Comment 4* </span>
<span style="color:blue;"> *Here I just added the information on the Cellosaurus IDs instead of initial indexing of cells if it is possible in `def prepare_cellinfo()`.*</span>

In [12]:
def clean_inst(inst: pd.DataFrame) -> pd.DataFrame:
    df = inst.copy()
    id_col = None
    for cand in ["inst_id", "sample_id", "distil_id"]:
        if cand in df.columns:
            id_col = cand
            break
    if id_col is None:
        raise KeyError("inst info missing inst_id/sample_id/distil_id")
    df = df.rename(columns={id_col: "lincs_inst_id"})
    # Comment: Added det_plate which could be derived from inst_id
    if not 'det_plate' in df.columns:
        df['det_plate'] = df['lincs_inst_id'].str.split(':').str[0]
    df = df.set_index("lincs_inst_id", drop=False)
    return df



def annotate_compounds(compound_df: pd.DataFrame) -> pd.DataFrame:
    comp = compound_df.copy()
    if "pert_id" in comp.columns:
        comp = comp.drop_duplicates(subset=["pert_id"]).set_index("pert_id")
    return comp



def prepare_cellinfo(cellinfo: pd.DataFrame,
                     cellinfo_extra: Optional[pd.DataFrame]  = None) -> pd.DataFrame:
    df = cellinfo.copy()
    cellinfo_id = 'cell_id'
    if cellinfo_extra is not None and not cellinfo_extra.empty:
        df_extra = cellinfo_extra.copy()
        df_extra[cellinfo_id] = df_extra['cell_iname'].str.replace('_', '.')
        df = df.merge(df_extra[[cellinfo_id, 'cellosaurus_id']], on=cellinfo_id, how='left')
        df[cellinfo_id  + '_mixed']  = df['cellosaurus_id'].fillna(df[cellinfo_id])
    rename_map = {col: f"cellinfo_{col}" for col in df.columns if col != "cell_id"}
    return df.rename(columns=rename_map).set_index("cell_id")



def standardize_dose(df: pd.DataFrame, compound_lookup: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dose = pd.to_numeric(out.get("pert_dose"), errors="coerce")
    unit = out.get("pert_dose_unit").astype("string").str.strip().str.lower()
    mw = pd.to_numeric(out.get("mw_g_per_mol"), errors="coerce") if "mw_g_per_mol" in out.columns else None
    dose_um = pd.Series(np.nan, index=out.index, dtype=float)
    mask_um = unit.isin({"um", "µm", "micromolar"})
    mask_nm = unit.isin({"nm", "nanomolar"})
    mask_mm = unit.isin({"mm", "millimolar"})
    mask_ng_ml = unit.isin({"ng/ml", "ng_per_ml"}) & (mw.notna() if mw is not None else False)
    dose_um[mask_um] = dose[mask_um]
    dose_um[mask_nm] = dose[mask_nm] / 1000.0
    dose_um[mask_mm] = dose[mask_mm] * 1000.0
    if mw is not None:
        dose_um[mask_ng_ml] = dose[mask_ng_ml] / mw[mask_ng_ml]
    out["pert_dose_um"] = dose_um
    return out



def standardize_time(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    time_val = pd.to_numeric(out.get("pert_time"), errors="coerce")
    time_unit = out.get("pert_time_unit").astype("string").str.strip().str.lower()
    hours = pd.Series(np.nan, index=out.index, dtype=float)
    hours[time_unit.isin({"h", "hr", "hrs", "hour", "hours"})] = time_val[time_unit.isin({"h", "hr", "hrs", "hour", "hours"})]
    hours[time_unit.isin({"d", "day", "days"})] = time_val[time_unit.isin({"d", "day", "days"})] * 24.0
    hours[time_unit.isin({"m", "min", "mins", "minute", "minutes"})] = time_val[time_unit.isin({"m", "min", "mins", "minute", "minutes"})] / 60.0
    out["pert_time_h"] = hours
    return out



def build_development_stage(row: pd.Series) -> str:
    for col in ["cellinfo_donor_age"]:
        if col in row.index and pd.notna(row[col]):
            try:
                age = float(row[col])
                if np.isfinite(age) and age > 0:
                    return f"{int(age)}-year-old stage"
            except ValueError:
                continue
    return "unknown"



def format_suspension(pattern: Optional[str]) -> str:
    if isinstance(pattern, str):
        val = pattern.strip().lower()
    elif pd.isna(pattern):
        return "unknown"
    else:
        val = str(pattern).strip().lower()
    if val in {"adherent", "suspension", "mixed"}:
        return val
    if "susp" in val:
        return "suspension"
    if "adh" in val:
        return "adherent"
    return "unknown"



def resolve_perturbagen(row: pd.Series) -> str:
    for col in ["perturbagen", "pert_iname", "pert_id", "compound_name", "cmap_name"]:
        if col in row.index and pd.notna(row[col]):
            text = str(row[col]).strip()
            if text and text.lower() not in {"nan", "-666"}:
                return text
    return "unknown"



def materialize_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    string_cols = df.select_dtypes(include="string").columns
    if len(string_cols):
        for col in string_cols:
            values = df[col].astype(object)
            df[col] = values.where(pd.notna(values), np.nan)
    if pd.api.types.is_string_dtype(getattr(df.index, "dtype", None)):
        df.index = df.index.astype(object)
    return df



## Build `.obs`

In [13]:
inst = clean_inst(inst_raw)

In [14]:
# Keep alternative identifier columns so we can match GCTX column IDs later
inst_raw_tmp = inst_raw.copy()

if 'lincs_inst_id' not in inst_raw_tmp.columns:
    if 'sample_id' in inst_raw_tmp.columns:
        inst_raw_tmp['lincs_inst_id'] = inst_raw_tmp['sample_id']
    elif 'inst_id' in inst_raw_tmp.columns:
        inst_raw_tmp['lincs_inst_id'] = inst_raw_tmp['inst_id']
    else:
        raise KeyError("instinfo is missing lincs_inst_id, sample_id, and inst_id columns.")

inst_raw_indexed = inst_raw_tmp.set_index('lincs_inst_id', drop=False)

In [15]:
for extra_id in ("distil_id", "inst_id", "sample_id", "sig_id"):
    if extra_id in inst_raw_indexed.columns:
        print(extra_id)
        inst[extra_id] = inst_raw_indexed.loc[inst.index, extra_id].astype("string")

inst_id


In [16]:
cellinfo = prepare_cellinfo(cellinfo_raw, cellinfo_extra=cellinfo_beta)
compound_lookup = annotate_compounds(compound_mw)

In [17]:
inst = inst.merge(cellinfo, how="left", left_on="cell_id", right_index=True)
inst = inst.merge(compound_lookup[["mw_g_per_mol"]], how="left", left_on="pert_id", right_index=True)
inst = inst.merge(pert_raw[["pert_id", "pert_iname", "pert_type", "pubchem_cid"]].drop_duplicates("pert_id"), how="left", on="pert_id", suffixes=("", "_pert"))

### <span style="color:blue;">*Comment 5* </span>
<span style="color:blue;"> *I like the original idea with the molecular weights. But currently as I see the compounds which molecular weights we know do not have units which we are interested in to convert (ng_per_ml). (At least in the current version of L1000)*</span>

<span style="color:blue;"> *Additionally, as I see no compound-like perturbagens have the units such as `ng/ml`. According to the [documentation](https://clue.io/connectopedia/perturbagen_types_and_controls) `trt_lig` related to the biological agents, not compounds*</span>

In [18]:
inst[~inst['mw_g_per_mol'].isna()]['pert_dose_unit'].unique()

array(['%', 'um', '-666'], dtype=object)

In [19]:
inst[inst['pert_dose_unit'] == 'ng/ml']['pert_type'].unique()

array(['trt_lig'], dtype=object)

In [20]:
inst = standardize_dose(inst, compound_lookup)
inst = standardize_time(inst)

if CONFIG["pert_types_to_keep"] is not None:
    inst = inst[inst["pert_type"].isin(CONFIG["pert_types_to_keep"])]
if CONFIG["ctl_perturbagens"] is not None:
    inst = inst[(inst['pert_type'].str.startswith('ctl') & inst['pert_iname'].isin(CONFIG['ctl_perturbagens']))\
        | (~inst['pert_type'].str.startswith('ctl'))]
if CONFIG["cell_types_to_keep"] is not None:
    inst = inst[inst["cell_id"].isin(CONFIG["cell_types_to_keep"])]
if CONFIG["perturbagens_to_keep"] is not None:
    inst = inst[inst["pert_id"].isin(CONFIG["perturbagens_to_keep"])]
if CONFIG["only_touchstone"] and "is_touchstone" in inst.columns:
    inst = inst[inst["is_touchstone"] == 1]
if CONFIG["min_dose_uM"] is not None:
    inst = inst[(inst["pert_dose_um"].isna()) | (inst["pert_dose_um"] >= CONFIG["min_dose_uM"])]
if CONFIG["max_dose_uM"] is not None:
    inst = inst[(inst["pert_dose_um"].isna()) | (inst["pert_dose_um"] <= CONFIG["max_dose_uM"])]
if CONFIG["time_h_min"] is not None:
    inst = inst[(inst["pert_time_h"].isna()) | (inst["pert_time_h"] >= CONFIG["time_h_min"])]
if CONFIG["time_h_max"] is not None:
    inst = inst[(inst["pert_time_h"].isna()) | (inst["pert_time_h"] <= CONFIG["time_h_max"])]

if CONFIG["max_samples"] is not None and len(inst) > CONFIG["max_samples"]:
    inst = inst.sample(CONFIG["max_samples"], random_state=0)

inst["source_gctx"] = str(PATHS["level2_gctx"])

In [21]:
obs = pd.DataFrame(index=inst.index)
# Carry over potential identifier columns for matching GCTX metadata later
for extra_id in ("lincs_inst_id", "distil_id", "inst_id", "sample_id", "sig_id"):
    if extra_id in inst.columns:
        obs[extra_id] = inst[extra_id].astype("string")

### <span style="color:blue;">*Comment 6* </span>
<span style="color:blue;"> ***1)** The values for column `plate` I took from `det_plate` which was derived from original `inst_id`*
</span>

<span style="color:blue;"> ***2)** Maybe we can use just a `pert_iname` column for `perturbagen` (according to the current scheme, we also can discuss on the standardization and add `pert_id` to make it possible to link with perturbagen meta info)* </span>

<span style="color:blue;"> ***3)** Maybe we need to discuss which annotation for `pert_type` is better to stick to as I am also unsure. Currently, I just set compound pert_type for all compound-based perturbations and DMSO vehicle.* </span>

<span style="color:blue;"> ***4)** I assume that the `uM` values for controls are `0`. Additionally, the same values for DMSO we have in other datasets.* </span>

<span style="color:blue;"> ***5)** We need to validate but I assume that the suspension type is `cell`* </span>

<span style="color:blue;"> ***6)** I would leave just `cellinfo_primary_site` to define `tissue`* </span>

<span style="color:blue;"> ***7)** Here I am not sure how to better define `tissue_type`. I think we need to discuss. According to the Tahoe and Sci-Plex datasets annotations by the Lamin lab, this column is defined as `cell culture`. Probably it would also suit L1000. Additionally, I like the field `cellinfo_cell_type` which contains the following unique values: `['cell line', 'primary', 'differentiated', nan, 'iPSC']`, maybe we need to leave it somewhere, I think it is quite informative.* </span>

<span style="color:blue;"> ***8)** Ideally, maybe it would worth to check if the names of concepts could correspond to names of MONDO terms (e.g. `MONDO:5233`); The same is for `self_reported_ethnicity` (to check the mapping on `ethnicity_ontology_term`, e.g. `HANCESTRO`)* </span>

<span style="color:blue;"> ***9)** I am not sure but I need to check what the library is here in the papers 👀* </span>

<span style="color:blue;"> ***10)** I suppose `stimulation` should be None for compound-related perturbations* </span>

<span style="color:blue;"> ***11)** I just mapped values to the certain format* </span>

<span style="color:blue;"> ***12)** Probably None values of `pubchem_cid` can be retrieved through PubChempy (But it would take some time to run)* </span>

<span style="color:blue;"> ***13)** Probably `psbulk_cells` should be `None`* </span>

<span style="color:blue;"> ***14)** Probably `psbulk_counts` should be `None`* </span>

<span style="color:blue;"> ***NB!** Additionally I modified the format of missing values; for the meta information: `'tissue', 'tissue_type', 'disease', 'development_stage', 'sex', 'self_reported_ethnicity'` I added `unknown` if we do not know it, in addition, if we have the missed info in other columns I added `None` for more convenient filtering on the later stages. Also, the columns with `int64` types have `-666` values instead of `None` as in this case it is meaningless info* </span>


</span>

In [22]:
#Comment 5, 1):
obs["plate"] = inst.get("det_plate", None)
obs["well"] = inst.get("rna_well", inst.get("det_well", None))
obs["cell_type"] = inst.get("cell_id_mixed", inst.get("cell_id", None))
#Comment 5, 2):
obs["perturbagen"] = inst.get("pert_iname", None)

#Comment 5, 3):
PERT_TYPE_MAP = {
    "trt_cp": "compound",
    "trt_lig": "ligand",
    "trt_misc": "misc",
    "trt_sh": "shrna",
    "trt_oe": "overexpression",
    "trt_xpr": "crispr",
    "ctl_vehicle": "compound"
}

obs["pert_type"] = inst["pert_type"].map(PERT_TYPE_MAP)
obs["is_control"] = np.where(inst["pert_type"].str.startswith("ctl"), True, False)
#Comment 5, 4):
obs["pert_dose_uM"] = inst["pert_dose_um"].astype(float)
obs.loc[obs['is_control'] == True, 'pert_dose_uM'] = 0
obs["pert_time_h"] = inst["pert_time_h"].astype(float)

#Comment 5, 5)
obs["suspension_type"] = "cell"

#Comment 5, 6)
obs["tissue"] = inst.get("cellinfo_primary_site", "unknown")
#Comment 5, 7)
obs["tissue_type"] = "cell culture"
#Comment 5, 8)
obs["disease"] = inst.get("cellinfo_subtype", "unknown")
#Comment 5, 9)
obs["library"] = LIBRARY_LABEL

#Comment 5, 10) 
obs["stimulation"] = None
obs["guide"] = None
obs["dataset"] = DATASET_LABEL
obs["assay"] = ASSAY_LABEL

obs["development_stage"] = inst.apply(build_development_stage, axis=1)
obs["organism"] = "human"

#Comment 5, 11) 
obs["sex"] = inst["cellinfo_donor_sex"].map({"M": "male", "F": "female"})

obs["self_reported_ethnicity"] = inst.get("cellinfo_donor_ethnicity", "unknown")

#Comment 5, 12)
obs["pubchem_cid"] = inst.get("pubchem_cid", None)
#Comment 5, 13)
obs["psbulk_cells"] = None
#Comment 5, 14)
obs["psbulk_counts"] = None

obs["lincs_inst_id"] = inst.index.astype("string")
obs["source_gctx"] = inst["source_gctx"].astype("string")
obs["sample_id"] = (
    obs["plate"].astype(str).str.replace(" ", "", regex=False) + "_" +
    obs["well"].astype(str).str.replace(" ", "", regex=False) + "_" +
    obs["perturbagen"].astype(str).str.replace(" ", "_", regex=False) + "_" +
    obs["cell_type"].astype(str).str.replace(" ", "_", regex=False)
)



In [23]:
obs = obs.replace({-666: None, '-666': None, 'None': None, 'nan': None, '<NA>': None})
obs = obs[~obs["sample_id"].duplicated(keep="first")]
obs = obs.set_index("inst_id", drop=True)
obs = materialize_string_columns(obs)
print(f"Prepared obs rows: {len(obs):,}")
obs.head()

Prepared obs rows: 699,298


,lincs_inst_id,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,suspension_type,tissue,tissue_type,disease,library,stimulation,guide,dataset,assay,development_stage,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts,source_gctx,sample_id
inst_id,,,,,,,,,,,,,,,,,,,,,,,,,,,
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:F13,0,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,F13,MCF7,DMSO,compound,True,0.0,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,None,None,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,None,None,None,/ictstr01/home/icb/olga.novitskaia/lincs_data/...,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_F13_DMSO_MCF7
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:G13,1,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,G13,MCF7,DMSO,compound,True,0.0,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,None,None,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,None,None,None,/ictstr01/home/icb/olga.novitskaia/lincs_data/...,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_G13_DMSO_MCF7
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:I13,2,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,I13,MCF7,DMSO,compound,True,0.0,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,None,None,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,None,None,None,/ictstr01/home/icb/olga.novitskaia/lincs_data/...,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_I13_DMSO_MCF7
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:K13,3,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,K13,MCF7,DMSO,compound,True,0.0,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,None,None,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,None,None,None,/ictstr01/home/icb/olga.novitskaia/lincs_data/...,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_K13_DMSO_MCF7
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:N13,4,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,N13,MCF7,DMSO,compound,True,0.0,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,None,None,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,None,None,None,/ictstr01/home/icb/olga.novitskaia/lincs_data/...,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_N13_DMSO_MCF7


### Cast to the strict schema

In [24]:
# obs.index = obs['inst_id']

In [25]:
cols_fillna_unknown = [
    'tissue',
    'tissue_type',
    'disease',
    'development_stage',
    'sex',
    'self_reported_ethnicity'
]
dtype_map = {col: dtype for col, dtype, _ in OBS_SCHEMA}
obs_for_schema = obs.copy()
for col, dtype in dtype_map.items():
    if col not in obs_for_schema.columns:
        obs_for_schema[col] = np.nan
    if dtype == "category":
        if col in cols_fillna_unknown:
            obs_for_schema[col] = (
                obs_for_schema[col]
                .fillna("unknown")
                .astype("string")
                .astype(object)
                .astype("category")
            )
        else:
            obs_for_schema[col] = (
                obs_for_schema[col]
                .astype(object)
                .astype("category")
            )
    elif dtype == "float64":
        obs_for_schema[col] = pd.to_numeric(obs_for_schema[col], errors="coerce")
    elif dtype == "int64":
        obs_for_schema[col] = (
            pd.to_numeric(obs_for_schema[col], errors="coerce")
            .fillna(-666)
            .astype("int64")
        )


In [26]:
obs_for_schema = obs_for_schema[obs_schema_df.index.tolist()]
obs_for_schema = materialize_string_columns(obs_for_schema)
obs_for_schema

,sample_id,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,suspension_type,tissue,tissue_type,disease,library,stimulation,guide,dataset,assay,development_stage,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts
inst_id,,,,,,,,,,,,,,,,,,,,,,,,,
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:F13,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_F13_DMSO_MCF7,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,F13,MCF7,DMSO,compound,True,0.00,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,NaN,NaN,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,NaN,-666,-666
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:G13,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_G13_DMSO_MCF7,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,G13,MCF7,DMSO,compound,True,0.00,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,NaN,NaN,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,NaN,-666,-666
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:I13,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_I13_DMSO_MCF7,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,I13,MCF7,DMSO,compound,True,0.00,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,NaN,NaN,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,NaN,-666,-666
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:K13,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_K13_DMSO_MCF7,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,K13,MCF7,DMSO,compound,True,0.00,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,NaN,NaN,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,NaN,-666,-666
ASG001_MCF7_24H_X1_B7_DUO52HI53LO:N13,ASG001_MCF7_24H_X1_B7_DUO52HI53LO_N13_DMSO_MCF7,ASG001_MCF7_24H_X1_B7_DUO52HI53LO,N13,MCF7,DMSO,compound,True,0.00,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,NaN,NaN,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,NaN,-666,-666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PCLB002_MCF7_24H_X3_B13:P20,PCLB002_MCF7_24H_X3_B13_P20_wortmannin_MCF7,PCLB002_MCF7_24H_X3_B13,P20,MCF7,wortmannin,compound,False,3.33,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,NaN,NaN,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,NaN,-666,-666
PCLB002_MCF7_24H_X3_B13:P21,PCLB002_MCF7_24H_X3_B13_P21_wortmannin_MCF7,PCLB002_MCF7_24H_X3_B13,P21,MCF7,wortmannin,compound,False,1.11,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,NaN,NaN,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,NaN,-666,-666
PCLB002_MCF7_24H_X3_B13:P22,PCLB002_MCF7_24H_X3_B13_P22_wortmannin_MCF7,PCLB002_MCF7_24H_X3_B13,P22,MCF7,wortmannin,compound,False,0.37,24.0,cell,breast,cell culture,adenocarcinoma,LINCS_L1000_Level2_epsilon,NaN,NaN,LINCS2017_level2,L1000,69-year-old stage,human,female,Caucasian,NaN,-666,-666


## Gene annotations (`.var`)

In [27]:

var = geneinfo_level2.copy()
var = var.rename(columns={
    "pr_gene_id": "gene_id",
    "pr_gene_symbol": "symbol",
    "pr_gene_title": "gene_title",
    "pr_is_lm": "is_landmark"
})
var = var.drop_duplicates(subset=["gene_id"]).set_index("gene_id")
var["symbol"] = var["symbol"].astype("string")
if geneinfo_beta is not None and {"gene_symbol", "ensembl_id"}.issubset(geneinfo_beta.columns):
    sym_to_ens = (
        geneinfo_beta[["gene_symbol", "ensembl_id"]]
        .dropna()
        .drop_duplicates(subset=["gene_symbol"])
        .set_index("gene_symbol")
        ["ensembl_id"]
    )
    var["ensembl_id"] = var["symbol"].map(sym_to_ens)
else:
    var["ensembl_id"] = var.index.astype("string")
# var["ensembl_id"] = var["ensembl_id"].fillna(var.index.astype("string"))
var["ensembl_id"] = var["ensembl_id"].fillna(var.index.to_series().astype("string"))
var = var[["symbol", "ensembl_id", "is_landmark"]]
if not FULL_GENE_MATRIX:
    landmark_mask = var["is_landmark"].fillna(0).astype(int)
    var = var.loc[landmark_mask == 1]
    print(f"Restricting to {len(var):,} landmark genes for expression.")
else:
    print(f"Using all {len(var):,} gene features (landmarks + inferred).")
var.head()


Restricting to 978 landmark genes for expression.


,symbol,ensembl_id,is_landmark
gene_id,,,
780,DDR1,ENSG00000204580,1
7849,PAX8,ENSG00000125618,1
6193,RPS5,ENSG00000083845,1
23,ABCF1,ENSG00000204574,1
9552,SPAG7,ENSG00000091640,1


## Expression extraction

In [28]:

from cmapPy.pandasGEXpress import parse
import h5py
import scipy.sparse as sp
import anndata as ad

FORCE_REBUILD_MEMMAP = False
_GCTX_COL_IDS = None
_GCTX_COL_ID_MAP = None

def _get_gctx_column_ids() -> set[str]:
    global _GCTX_COL_IDS, _GCTX_COL_ID_MAP
    if _GCTX_COL_IDS is None:
        gctx_path = PATHS["level2_gctx"]
        if not gctx_path.exists():
            raise FileNotFoundError(gctx_path)
        with h5py.File(gctx_path, "r") as handle:
            root = handle["0"] if "0" in handle else handle
            ids = root["META"]["COL"]["id"][:]
        _GCTX_COL_IDS = [
            id_.decode("utf-8") if isinstance(id_, bytes) else str(id_)
            for id_ in ids
        ]
        _GCTX_COL_ID_MAP = {id_.lower(): id_ for id_ in _GCTX_COL_IDS}
        print(f"Loaded {len(_GCTX_COL_IDS):,} column IDs from {gctx_path.name}.")
    return set(_GCTX_COL_IDS)

def _filter_obs_for_gctx(obs_subset: pd.DataFrame) -> pd.DataFrame:
    available = _get_gctx_column_ids()
    mask = obs_subset["gctx_id"].isin(available)
    if not mask.all():
        dropped = int((~mask).sum())
        print(f"⚠️ Dropping {dropped} samples not present in the Level 2 matrix.")
    obs_filtered = obs_subset.loc[mask].copy()
    if obs_filtered.empty:
        raise ValueError("No samples remain after intersecting with the Level 2 matrix.")
    return obs_filtered

def _normalize_gene_ids(gene_ids: Optional[Iterable[str]]) -> Optional[list[str]]:
    if gene_ids is None:
        return None
    return list(dict.fromkeys(str(gid) for gid in gene_ids))


def write_memmap_from_level2(obs_subset: pd.DataFrame, gene_ids: Optional[Iterable[str]] = None) -> None:
    if EXPR_MEMMAP.exists() and not FORCE_REBUILD_MEMMAP:
        print("Memmap already exists; reuse it unless FORCE_REBUILD_MEMMAP=True")
        return
    obs_subset = _filter_obs_for_gctx(obs_subset)
    gene_ids = _normalize_gene_ids(gene_ids)
    lincs_ids = obs_subset["gctx_id"].tolist()
    gctx = parse.parse(
        str(PATHS["level2_gctx"]),
        cid=lincs_ids,
        rid=gene_ids,
    )
    expr = gctx.data_df.loc[:, lincs_ids]
    expr.columns = obs_subset.index
    np.save(EXPR_SAMPLE_IDS, expr.columns.to_numpy(str))
    np.save(EXPR_GENE_IDS, expr.index.to_numpy(str))
    np.save(EXPR_MEMMAP, expr.T.to_numpy(np.float32))
    print(f"Saved memmap with shape {expr.shape[1]} × {expr.shape[0]} (samples × genes)")

def load_expression(obs_subset: pd.DataFrame, gene_ids: Optional[Iterable[str]] = None) -> pd.DataFrame:
    obs_subset = _filter_obs_for_gctx(obs_subset)
    gene_ids = _normalize_gene_ids(gene_ids)
    if (
        EXPR_MEMMAP.exists()
        and EXPR_SAMPLE_IDS.exists()
        and EXPR_GENE_IDS.exists()
        and not FORCE_REBUILD_MEMMAP
    ):
        sample_ids = np.load(EXPR_SAMPLE_IDS, allow_pickle=False).astype(str)
        gene_ids_mem = np.load(EXPR_GENE_IDS, allow_pickle=False).astype(str)
        X_mem = np.load(EXPR_MEMMAP, mmap_mode="r")
        expr_df = pd.DataFrame(X_mem, index=sample_ids, columns=gene_ids_mem)
        expr_df = expr_df.reindex(obs_subset.index)
        if gene_ids is not None:
            expr_df = expr_df.loc[:, gene_ids]
        return expr_df
    write_memmap_from_level2(obs_subset, gene_ids)
    return load_expression(obs_subset, gene_ids)

def build_obs_helpers(obs: pd.DataFrame) -> pd.DataFrame:
    gctx_ids = _get_gctx_column_ids()
    id_map = _GCTX_COL_ID_MAP or {}
    index_vals = obs.index.astype("string")
    mask = index_vals.isin(gctx_ids)
    if mask.any():
        matched = int(mask.sum())
        print(f"Using `index` to subset GCTX ({matched}/{len(obs)} samples match).")
        helpers = obs.loc[mask, ["source_gctx"]].copy()
        helpers["gctx_id"] = index_vals[mask]
        return helpers
    # Optional: case-insensitive fallback
    lower_vals = index_vals.str.lower()
    mask_lower = lower_vals.isin(id_map)
    if mask_lower.any():
        matched = int(mask_lower.sum())
        print(f"Using `index` (case-insensitive) to subset GCTX ({matched}/{len(obs)} samples match).")
        helpers = obs.loc[mask_lower, ["source_gctx"]].copy()
        helpers["gctx_id"] = lower_vals[mask_lower].map(id_map)
        return helpers
    raise ValueError("None of the obs ID columns match the Level 2 GCTX metadata.")





In [29]:
obs_helpers = build_obs_helpers(obs)
gene_ids_for_expr = var.index.astype(str)
expr_df = load_expression(obs_helpers, gene_ids=gene_ids_for_expr)
expr_df = expr_df.reindex(columns=gene_ids_for_expr)

print(expr_df.shape)

Loaded 1,269,922 column IDs from GSE92742_Broad_LINCS_Level2_GEX_epsilon_n1269922x978.gctx.
Using `index` to subset GCTX (697709/699298 samples match).
(697709, 978)


In [30]:
# print how many columns in expr_df are not all NaN 
print(f"Number of genes with at least one non-NaN value: {len(expr_df.columns[~expr_df.isna().all()]):,}")

Number of genes with at least one non-NaN value: 978


In [31]:
import gc

del _GCTX_COL_IDS
del _GCTX_COL_ID_MAP
del obs
del obs_helpers
del inst
del inst_raw_tmp
del inst_raw_indexed
del compound_lookup
del cellinfo
del geneinfo_beta
del geneinfo_level2
del pert_raw
gc.collect()

96

## Assemble AnnData

In [32]:
var_df = var.reindex(expr_df.columns.astype(int)).copy()
# var_idx = var_df["ensembl_id"].fillna(var_df.index.astype("string"))
var_idx = var_df["ensembl_id"].fillna(var_df.index.to_series().astype("string"))
var_idx = var_idx.astype(object)

### <span style="color:blue;">*Comment 7* </span>
<span style="color:blue;"> *I also encountered the problem with the None values in ensemble_ids, duplicates in ens_IDs as well as I did not found the release of the Ensembl which is used in `geneinfo_beta` file. I need to think how to map gene ids to ensembl in this case. (According to the documentation genes are represented as Entrez ID, HUGO and Ensembl in l1000). Original int id is Entrez id. Maybe we can take a mixture of IDs, but I am not sure here*</span>

In [33]:
duplicate_mask = var_idx.duplicated(keep=False)
if duplicate_mask.any():
    suffix = (
        var_idx[duplicate_mask]
        .groupby(var_idx[duplicate_mask])
        .cumcount()
        .astype("string")
    )
    var_idx = var_idx.astype("string")
    var_idx[duplicate_mask] = var_idx[duplicate_mask] + "_" + suffix
var_idx = var_idx.astype(object)
var_df.index = var_idx
var_df = var_df[["symbol"]]

var_df["symbol"] = (
    var_df["symbol"]
    .replace({"": None})
    .astype(object)
    .astype("category")
    )

var_df = materialize_string_columns(var_df)

obs_final = obs_for_schema.loc[expr_df.index].set_index('sample_id')

In [34]:
adata = ad.AnnData(X=sp.csr_matrix(expr_df.to_numpy(np.float32)), obs=obs_final, var=var_df)
print(adata)

AnnData object with n_obs × n_vars = 697709 × 978
    obs: 'plate', 'well', 'cell_type', 'perturbagen', 'pert_type', 'is_control', 'pert_dose_uM', 'pert_time_h', 'suspension_type', 'tissue', 'tissue_type', 'disease', 'library', 'stimulation', 'guide', 'dataset', 'assay', 'development_stage', 'organism', 'sex', 'self_reported_ethnicity', 'pubchem_cid', 'psbulk_cells', 'psbulk_counts'
    var: 'symbol'


In [35]:

print("obs columns:", list(adata.obs.columns))
print("var columns:", list(adata.var.columns))
print("Top cell types:")
display(adata.obs["cell_type"].value_counts().head(10))
print("Top perturbagens:")
display(adata.obs["perturbagen"].value_counts().head(10))


obs columns: ['plate', 'well', 'cell_type', 'perturbagen', 'pert_type', 'is_control', 'pert_dose_uM', 'pert_time_h', 'suspension_type', 'tissue', 'tissue_type', 'disease', 'library', 'stimulation', 'guide', 'dataset', 'assay', 'development_stage', 'organism', 'sex', 'self_reported_ethnicity', 'pubchem_cid', 'psbulk_cells', 'psbulk_counts']
var columns: ['symbol']
Top cell types:


cell_type
VCAP      123779
MCF7      103291
PC3        95887
A549       60548
A375       54903
HT29       53369
HA1E       32532
HCC515     27383
HEPG2      18929
NPC        15060
Name: count, dtype: int64

Top perturbagens:


perturbagen
DMSO              27102
vorinostat         4175
trichostatin-a     3677
wortmannin         3263
geldanamycin       3140
sirolimus          1478
curcumin           1388
sulforaphane       1195
fulvestrant        1194
genistein          1189
Name: count, dtype: int64

In [36]:
OUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(OUT_H5AD, compression="gzip")
print(f" Saved {adata.n_obs:,} × {adata.n_vars:,} AnnData → {OUT_H5AD}")

 Saved 697,709 × 978 AnnData → /ictstr01/home/icb/olga.novitskaia/lincs_data/processed/l1000_level2_deg_ready_landmark.h5ad
